In [2]:
import boto3
from botocore import UNSIGNED
from botocore.client import Config


s3 = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED)
)

In [7]:
bucket = "kumo-public-datasets"
prefix = "hm_with_images/"

In [ ]:


# paginator = s3.get_paginator("list_objects_v2")

# all_keys = []

# for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
#     for obj in page.get("Contents", []):
#         all_keys.append(obj["Key"])

In [5]:
# output_file = "bucket_content.txt"

# with open(output_file, "w", encoding="utf-8") as f:
#     for key in all_keys:
#         f.write(key + "\n")

# print(f"Saved {len(all_keys)} objects to {output_file}")

In [8]:
from collections import defaultdict
paginator = s3.get_paginator("list_objects_v2")

groups = defaultdict(list)

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        
        if not key.endswith(".jpg"):
            continue
        
        # extract "010" from images/010/xxx.jpg
        parts = key.split("/")
        if len(parts) < 3:
            continue
        
        n_str = parts[2]  # "010"
        
        if n_str.isdigit():
            n = int(n_str)
            groups[n].append(key)

In [9]:
groups

defaultdict(list,
            {10: ['hm_with_images/images/010/0108775015.jpg',
              'hm_with_images/images/010/0108775044.jpg',
              'hm_with_images/images/010/0108775051.jpg'],
             11: ['hm_with_images/images/011/0110065001.jpg',
              'hm_with_images/images/011/0110065002.jpg',
              'hm_with_images/images/011/0110065011.jpg',
              'hm_with_images/images/011/0111565001.jpg',
              'hm_with_images/images/011/0111565003.jpg',
              'hm_with_images/images/011/0111586001.jpg',
              'hm_with_images/images/011/0111593001.jpg',
              'hm_with_images/images/011/0111609001.jpg',
              'hm_with_images/images/011/0112679048.jpg',
              'hm_with_images/images/011/0112679052.jpg',
              'hm_with_images/images/011/0114428026.jpg',
              'hm_with_images/images/011/0114428030.jpg',
              'hm_with_images/images/011/0116379047.jpg',
              'hm_with_images/images/011/0118

## Only sampling data from the first two groups for simplicity

In [10]:
valid_ns = [n for n in groups.keys() ][:2]

In [11]:
valid_ns

[10, 11]

In [12]:
import random
sampled = {}

for n in valid_ns:
    imgs = groups[n]
    
    if len(imgs) == 0:
        continue

    k = len(imgs)  
    
    sampled[n] = random.sample(imgs, k)

In [13]:
sampled

{10: ['hm_with_images/images/010/0108775044.jpg',
  'hm_with_images/images/010/0108775015.jpg',
  'hm_with_images/images/010/0108775051.jpg'],
 11: ['hm_with_images/images/011/0116379047.jpg',
  'hm_with_images/images/011/0111565003.jpg',
  'hm_with_images/images/011/0118458034.jpg',
  'hm_with_images/images/011/0118458029.jpg',
  'hm_with_images/images/011/0112679048.jpg',
  'hm_with_images/images/011/0118458004.jpg',
  'hm_with_images/images/011/0118458039.jpg',
  'hm_with_images/images/011/0110065011.jpg',
  'hm_with_images/images/011/0111609001.jpg',
  'hm_with_images/images/011/0110065002.jpg',
  'hm_with_images/images/011/0118458003.jpg',
  'hm_with_images/images/011/0114428030.jpg',
  'hm_with_images/images/011/0111593001.jpg',
  'hm_with_images/images/011/0111565001.jpg',
  'hm_with_images/images/011/0111586001.jpg',
  'hm_with_images/images/011/0112679052.jpg',
  'hm_with_images/images/011/0118458028.jpg',
  'hm_with_images/images/011/0114428026.jpg',
  'hm_with_images/images/

In [14]:
import os
os.makedirs("sampled_images", exist_ok=True)

for n, keys in sampled.items():
    out_dir = f"sampled_images/{n:03d}"
    os.makedirs(out_dir, exist_ok=True)
    
    for key in keys:
        filename = key.split("/")[-1]
        local_path = os.path.join(out_dir, filename)
        
        s3.download_file(bucket, key, local_path)

In [15]:
# c
sampled
s3.download_file(bucket, "hm_with_images/customers/_SUCCESS", "customers_SUCCESS")
s3.download_file(bucket, "hm_with_images/articles/_SUCCESS", "articles_SUCCESS")

In [16]:
from pyarrow import parquet as pq
from io import BytesIO

def read_bucket_parquet(key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    data = obj["Body"].read()
    return pq.read_table(BytesIO(data))
articles_key = "hm_with_images/articles/part-00000-63ea08b0-f43e-48ff-83ad-d1b7212d7840-c000.snappy.parquet"
customers_key = "hm_with_images/customers/part-00000-9b749c0f-095a-448e-b555-cbfb0bb7a01c-c000.snappy.parquet"
transactions_key = "hm_with_images/transactions/part-00000-d069d8ca-004d-4bc6-a901-ce9a2cd13c83-c000.snappy.parquet"



In [17]:
articles_df =read_bucket_parquet(articles_key).to_pandas()

In [18]:
articles_df.to_csv("articles_df_full.csv", index=False)

In [6]:
customers_df =read_bucket_parquet(customers_key).to_pandas()

In [7]:
customers_df.to_csv("customers_df_full.csv", index=False)

In [19]:
transactions_df = read_bucket_parquet(transactions_key).to_pandas()

In [20]:
transactions_df.to_csv("transactions_df_full.csv", index=False)

In [21]:
transactions_df[transactions_df["article_id"] == 108775044]

,article_id,customer_id,t_dat,price,sales_channel_id,transaction_id,analysis_column_30,analysis_column_90
12555,108775044,060fccc123592dd682ba60e95da1e2e75ebf1514abe50e...,2019-01-12,0.008458,2,17184750534,6,14
12556,108775044,060fccc123592dd682ba60e95da1e2e75ebf1514abe50e...,2019-01-12,0.008458,2,17184750535,6,14
17829,108775044,09030aebc7121fbc6f973a82ba8f42626549e54c7af519...,2019-05-04,0.008458,1,17189658738,0,9
22490,108775044,0bcc07f9a39fe21277d8ca70eb03208f34b9b3bb994701...,2019-05-07,0.007034,2,17189792692,0,18
22491,108775044,0bcc07f9a39fe21277d8ca70eb03208f34b9b3bb994701...,2019-05-07,0.007051,2,17189792693,0,18
...,...,...,...,...,...,...,...,...
465550,108775044,f9f01fea85bb4ce7062400df7dec056f1ca93760e4ff35...,2020-03-26,0.008458,2,17203634124,0,29
467037,108775044,facfbd96dc3a557f4b7ecb316436cb39b14ac501e14a7e...,2019-05-14,0.008458,2,17190110296,0,0
467038,108775044,facfbd96dc3a557f4b7ecb316436cb39b14ac501e14a7e...,2019-05-14,0.008458,2,17190110297,0,0
467039,108775044,facfbd96dc3a557f4b7ecb316436cb39b14ac501e14a7e...,2019-05-14,0.008458,2,17190110298,0,0


In [22]:
articles_df.head()

,article_id,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,image_url
0,108775015,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,https://kumo-public-datasets.s3.us-west-2.amaz...
1,108775044,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,https://kumo-public-datasets.s3.us-west-2.amaz...
2,108775051,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,https://kumo-public-datasets.s3.us-west-2.amaz...
3,110065001,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear",https://kumo-public-datasets.s3.us-west-2.amaz...
4,110065002,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear",https://kumo-public-datasets.s3.us-west-2.amaz...


In [23]:
articles_df.columns

Index(['article_id', 'prod_name', 'product_type_no', 'product_type_name',
       'product_group_name', 'graphical_appearance_no',
       'graphical_appearance_name', 'colour_group_code', 'colour_group_name',
       'perceived_colour_value_id', 'perceived_colour_value_name',
       'perceived_colour_master_id', 'perceived_colour_master_name',
       'department_no', 'department_name', 'index_code', 'index_name',
       'index_group_no', 'index_group_name', 'section_no', 'section_name',
       'garment_group_no', 'garment_group_name', 'image_url'],
      dtype='object')

## Exploring images
### (Decided not to include them in the demo given the limited time)

In [25]:
sampled_images_dir = "sampled_images"
directory_names = os.listdir(sampled_images_dir)
article_ids = []
for directory_name in directory_names:
    directory_path = os.path.join(sampled_images_dir, directory_name)
    if os.path.isdir(directory_path):
        file_names = os.listdir(directory_path)
        for file_name in file_names:
            if file_name.endswith(".jpg"):
                article_id = file_name.split(".")[0]
                if article_id[0] == "0":
                    article_id = article_id[1:]
                article_ids.append(int(article_id))
        





In [26]:
article_ids = list(set(article_ids))

In [27]:
article_ids

[108775044,
 112679048,
 108775051,
 112679052,
 118458003,
 118458004,
 116379047,
 111593001,
 111609001,
 118458028,
 118458029,
 118458034,
 118458038,
 118458039,
 130035001,
 111565001,
 129085001,
 111565003,
 126589006,
 111586001,
 108775015,
 110065001,
 110065002,
 110065011,
 114428026,
 114428030]

In [28]:
transactions_sampled = transactions_df[transactions_df["article_id"].isin(article_ids)]

In [29]:
transactions_sampled.head()

,article_id,customer_id,t_dat,price,sales_channel_id,transaction_id,analysis_column_30,analysis_column_90
2384,111565001,012cfe5d988e9635b9260b5128e7432aebb72467782f1c...,2019-01-03,0.008458,2,17184343146,0,3
2385,111565001,012cfe5d988e9635b9260b5128e7432aebb72467782f1c...,2019-01-03,0.008458,2,17184343147,0,3
2563,111586001,01429271bdb41f2d190e22be736a054d186a4f370181e7...,2019-11-05,0.016932,1,17198541454,0,7
3567,111593001,017836bae9f36aa127ac010640a39a24a2a159e4795d95...,2018-10-12,0.013542,2,17180911938,1,11
3600,111593001,017836bae9f36aa127ac010640a39a24a2a159e4795d95...,2019-03-22,0.011288,1,17187529929,8,22


In [30]:
len(transactions_sampled)

960

In [31]:
transactions_sampled['price'] = transactions_sampled['price'].astype(float) *1000

C:\Users\vsams\AppData\Local\Temp\ipykernel_33596\2588780684.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transactions_sampled['price'] = transactions_sampled['price'].astype(float) *1000


## Transactions with price adjusted

In [33]:
transactions_sampled.head()

,article_id,customer_id,t_dat,price,sales_channel_id,transaction_id,analysis_column_30,analysis_column_90
2384,111565001,012cfe5d988e9635b9260b5128e7432aebb72467782f1c...,2019-01-03,8.457627,2,17184343146,0,3
2385,111565001,012cfe5d988e9635b9260b5128e7432aebb72467782f1c...,2019-01-03,8.457627,2,17184343147,0,3
2563,111586001,01429271bdb41f2d190e22be736a054d186a4f370181e7...,2019-11-05,16.932203,1,17198541454,0,7
3567,111593001,017836bae9f36aa127ac010640a39a24a2a159e4795d95...,2018-10-12,13.542373,2,17180911938,1,11
3600,111593001,017836bae9f36aa127ac010640a39a24a2a159e4795d95...,2019-03-22,11.288136,1,17187529929,8,22


In [ ]:
import pandas as pd
transactions_sampled_articles = pd.merge(transactions_sampled, articles_df, on="article_id", how="left")
transactions_sampled_articles.head()
transactions_sampled_articles.columns
transactions_sampled_articles = transactions_sampled_articles[["article_id", "t_dat", "price", "prod_name"]]
transactions_sampled_articles.head()


,article_id,t_dat,price,prod_name
0,111565001,2019-01-03,8.457627,20 den 1p Stockings
1,111565001,2019-01-03,8.457627,20 den 1p Stockings
2,111586001,2019-11-05,16.932203,Shape Up 30 den 1p Tights
3,111593001,2018-10-12,13.542373,Support 40 den 1p Tights
4,111593001,2019-03-22,11.288136,Support 40 den 1p Tights


In [36]:
transactions_sampled_articles.to_csv("transactions_sampled_articles.csv", index=False)